# CLIP zero-shot Evaluation
This short notebook implements the dataset split into base and novel categories (see project assignment) and runs the zero-shot evaluation with CLIP.
Feel free to copy the code contained in this notebook or to directly use this notebook as starting point for you project.

In [ ]:
import torch
import torchvision
import clip
from tqdm import tqdm

## Dataset Loading
Let's get the data directly from torchvision as we have seen during labs.

In [16]:
def get_data(data_dir="./data", transform=None):
    """Load Flowers102 train, validation and test sets.
    Args:
        data_dir (str): Directory where the dataset will be stored.
        transform (torch.Compose)
    Returns:
        tuple: A tuple containing the train, validation, and test sets.
    """
    train = torchvision.datasets.Flowers102(root=data_dir, split="train", download=True, transform=transform)
    val = torchvision.datasets.Flowers102(root=data_dir, split="val", download=True, transform=transform)
    test = torchvision.datasets.Flowers102(root=data_dir, split="test", download=True, transform=transform)
    return train, val, test

## Base and Novel categories
To split in base and novel categories we list all dataset classes, and count their number (we already know it's 102 but let's do it properly).
Then, we just allocate the first half to base categories and the remaining half to novel ones.
We can do this because we are simulating a real world application, but keep in mind this will not happen out there!

In [17]:
def base_novel_categories(dataset):
    # set returns the unique set of all dataset classes
    all_classes = set(dataset._labels)
    # and let's count them
    num_classes = len(all_classes)

    # here list(range(num_classes)) returns a list from 0 to num_classes - 1
    # then we slice the list in half and generate base and novel category lists
    base_classes = list(range(num_classes))[:num_classes//2]
    novel_classes = list(range(num_classes))[num_classes//2:]
    return base_classes, novel_classes

## Inspect Classes
Let's now visualize which are the base and novel classes.
To do so, we first get a dummy test set (without augmentations) as we are just interested in the dataset labels. Then, we split it useing `base_novel_categories`.
Finally, we use the hard-coded CLASS_NAMES to print the class in natural language.

> Note: the list of class names was only recently added to `torchvision.datasets.Flowers102`. To avoid useless errors that can occour to you, we decided to also provide such a list.

In [18]:
_, _, tmp_test = get_data()
base_classes, novel_classes = base_novel_categories(tmp_test)
CLASS_NAMES = ["pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea", "english marigold", "tiger lily", "moon orchid", "bird of paradise", "monkshood", "globe thistle", "snapdragon", "colt's foot", "king protea", "spear thistle", "yellow iris", "globe-flower", "purple coneflower", "peruvian lily", "balloon flower", "giant white arum lily", "fire lily", "pincushion flower", "fritillary", "red ginger", "grape hyacinth", "corn poppy", "prince of wales feathers", "stemless gentian", "artichoke", "sweet william", "carnation", "garden phlox", "love in the mist", "mexican aster", "alpine sea holly", "ruby-lipped cattleya", "cape flower", "great masterwort", "siam tulip", "lenten rose", "barbeton daisy", "daffodil", "sword lily", "poinsettia", "bolero deep blue", "wallflower", "marigold", "buttercup", "oxeye daisy", "common dandelion", "petunia", "wild pansy", "primula", "sunflower", "pelargonium", "bishop of llandaff", "gaura", "geranium", "orange dahlia", "pink-yellow dahlia?", "cautleya spicata", "japanese anemone", "black-eyed susan", "silverbush", "californian poppy", "osteospermum", "spring crocus", "bearded iris", "windflower", "tree poppy", "gazania", "azalea", "water lily", "rose", "thorn apple", "morning glory", "passion flower", "lotus", "toad lily", "anthurium", "frangipani", "clematis", "hibiscus", "columbine", "desert-rose", "tree mallow", "magnolia", "cyclamen", "watercress", "canna lily", "hippeastrum", "bee balm", "ball moss", "foxglove", "bougainvillea", "camellia", "mallow", "mexican petunia", "bromelia", "blanket flower", "trumpet creeper", "blackberry lily"]
print("Base Class Names:", [(i, CLASS_NAMES[i]) for i in base_classes])
print("Novel Class Names:", [(i, CLASS_NAMES[i]) for i in novel_classes])

Base Class Names: [(0, 'pink primrose'), (1, 'hard-leaved pocket orchid'), (2, 'canterbury bells'), (3, 'sweet pea'), (4, 'english marigold'), (5, 'tiger lily'), (6, 'moon orchid'), (7, 'bird of paradise'), (8, 'monkshood'), (9, 'globe thistle'), (10, 'snapdragon'), (11, "colt's foot"), (12, 'king protea'), (13, 'spear thistle'), (14, 'yellow iris'), (15, 'globe-flower'), (16, 'purple coneflower'), (17, 'peruvian lily'), (18, 'balloon flower'), (19, 'giant white arum lily'), (20, 'fire lily'), (21, 'pincushion flower'), (22, 'fritillary'), (23, 'red ginger'), (24, 'grape hyacinth'), (25, 'corn poppy'), (26, 'prince of wales feathers'), (27, 'stemless gentian'), (28, 'artichoke'), (29, 'sweet william'), (30, 'carnation'), (31, 'garden phlox'), (32, 'love in the mist'), (33, 'mexican aster'), (34, 'alpine sea holly'), (35, 'ruby-lipped cattleya'), (36, 'cape flower'), (37, 'great masterwort'), (38, 'siam tulip'), (39, 'lenten rose'), (40, 'barbeton daisy'), (41, 'daffodil'), (42, 'sword 

## Split Dataset
The next step is to actually split the dataset into the base and novel categories we extract from `base_novel_categories`.
To split the data we need the dataset (obviously) and the list of base classes. If the sample label is not part of the base categories, then it must be part of the novel ones.

In [19]:
def split_data(dataset, base_classes):
    # these two lists will store the sample indexes
    base_categories_samples = []
    novel_categories_samples = []

    # we create a set of base classes to compute the test below in O(1)
    # this is optional and can be removed
    base_set = set(base_classes)

    # here we iterate over sample labels and also get the correspondent sample index
    for sample_id, label in enumerate(dataset._labels):
        if label in base_set:
            base_categories_samples.append(sample_id)
        else:
            novel_categories_samples.append(sample_id)

    # here we create the dataset subsets
    # the torch Subset is just a wrapper around the dataset
    # it simply stores the subset indexes and the original dataset (your_subset.dataset)
    # when asking for sample i in the subset, torch will look for its original position in the dataset and retrieve it
    # https://pytorch.org/docs/stable/data.html#torch.utils.data.Subset
    base_dataset = torch.utils.data.Subset(dataset, base_categories_samples)
    novel_dataset = torch.utils.data.Subset(dataset, novel_categories_samples)
    return base_dataset, novel_dataset

## Extract k shots
As the dataset already provides 10 train and validation shots, we do not need to extract them.
Beaware that Few-Shot Adaptation papers must do this operation as most datasets count significantly more samples in both the training and validation sets.

## Load CLIP

In [20]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# available models = ['RN50', 'RN101', 'RN50x4', 'RN50x16', 'RN50x64', 'ViT-B/32', 'ViT-B/16', 'ViT-L/14', 'ViT-L/14@336px']
model, preprocess = clip.load("ViT-B/16", device="cpu")
model = model.to(device)

# preprocess contains CLIP's pre-defined augmentations, let's inspect them!
preprocess

Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    <function _convert_image_to_rgb at 0x7188ca8e9120>
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)

## Load and Prepare Data
Here we get the three dataset split and pass clip pre-defined augmentations.
Then, we compute base and novel categories (in this case is redundand as we already did it before).
Finally, se split the three datasets into base and novel categories.
As we want to use the novel categories only for the test set, we drop `train_novel` and `val_novel`.

In [21]:
# get the three datasets
train_set, val_set, test_set = get_data(transform=preprocess)

# split classes into base and novel
base_classes, novel_classes = base_novel_categories(train_set)

# split the three datasets
train_base, _ = split_data(train_set, base_classes)
val_base, _ = split_data(val_set, base_classes)
test_base, test_novel = split_data(test_set, base_classes)

In [ ]:
colors = [
"Red", "Crimson Red", "Dark Red", "Electric Red", "Maroon", "Scarlet", "Tuscan Red", "Pink", "Amaranth", "Bright Magenta", "Bubblegum Pink",
"Deep Pink", "Hot Pink", "Luminescent Pink", "Magenta", "Raspberry Pink", "Rose", "Salmon Pink", "Orange", "Apricot Orange", "Bright Orange",
"Coral Orange", "Orange Red", "Peachy", "Red-orange", "Brown", "Burning Gold", "Bronze", "Beige", "Chocolate Brown", "Coffee",
"Copper", "Ochre", "Sepia", "Terracotta", "Yellow", "Amber", "Cream", "Gold", "Khaki", "Lemon Yellow", "Light Yellow", "Lively Yellow",
"Marigold Yellow", "Mustard Yellow", "Green", "Aquamarine", "Green-yellow", "Dark Green", "Electric lime", "Forest Green",
"Jade Green", "Lime Green", "Mint Green", "Olive Green", "Spring Green", "Teal Blue", "Blue", "Abyssal Blue", "Aqua", "Aquamarine",
"Blue Gray", "Celeste", "Cobalt blue", "Cyan", "Dark Blue", "Dark Turquoise", "Deep Blue", "Deep Sky Blue", "Denim Blue", "Electric Blue",
"Ice Blue", "Light Blue", "Light Sky Blue", "Medium Blue", "Medium Turquoise", "Midnight Blue", "Navy Blue", "Neon Blue", "Pale Turquoise", "Royal Blue",
"Sapphire Blue", "Sky Blue", "Tiffany Blue", "Turquoise", "Ultramarine", "Purple", "Amethyst Purple", "Bright Violet", "Electric Violet",
"Indigo", "Lavender", "Lilac", "Obsidian Purple", "Strong Violet", "Violet", "Black", "Eerie Black", "Gray", "Dark Gray", "Dull Gray",
"Light Gray", "White", "Azure Mist", "Ivory", "Platinum", "Snow White", "White"
]

In [ ]:
colors = [
"Red", "Light Pink", "Pink", "Dark Pink", "Magenta", "Orange", "Brown", "Yellow", "Green", "Blue", "Aqua", "Blue Gray", "Celeste", "Cyan", "Dark Blue", "Electric Blue",
"Light Blue", "Navy Blue", "Purple", "Violet", "Lilac", "Lavender", "Indigo", "Black", "Gray", "Light Gray", "White", "Creamy White", "Pearly White"
]

## Compute Zero-Shot Predictions

In [24]:
@torch.no_grad()
def eval_colors(model, dataset, categories, batch_size, device, label=""):
    model.eval()

    # Create text inputs for colors
    color_text_inputs = clip.tokenize(
        [f"a photo of a {color} flower." for color in colors]  # Fixed: use 'color' not 'c'
    ).to(device)

    # Encode text features for colors
    text_features = model.encode_text(color_text_inputs)  # Fixed: use color_text_inputs
    text_features /= text_features.norm(dim=-1, keepdim=True)

    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    found_colors = []

    for image, target in tqdm(dataloader, desc=label):
        image = image.to(device)

        # Forward image through CLIP image encoder
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)

        # Get predicted color indices
        predicted_color_indices = (image_features @ text_features.T).argmax(dim=-1)

        # Convert indices to actual color names
        batch_colors = [colors[idx.item()] for idx in predicted_color_indices]
        found_colors.extend(batch_colors)  # Use extend to add all colors from batch

    return found_colors  # Return the list of predicted color names

found_colors = eval_colors(model=model, dataset=test_base, categories=base_classes, batch_size=128, device=device, label="Finding color clues")
novel_found_colors = eval_colors(model=model, dataset=test_novel, categories=novel_classes, batch_size=128, device=device, label="Finding color clues")
print("hello world")

Finding color clues: 100%|██████████| 29/29 [01:21<00:00,  2.80s/it]

hello world


In [25]:
len(found_colors), len(novel_found_colors), len(test_base), len(test_novel)

(2473, 3676, 2473, 3676)

In [ ]:
@torch.no_grad() # we don't want gradients
def eval(model, dataset, categories, batch_size, device, found_colors, label=""):
    # let's set the model in evaluation mode
    model.eval()

    # Remap labels into a contiguous set starting from zero
    contig_cat2idx = {cat: idx for idx, cat in enumerate(categories)}

    # simple dataloader creation
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # here we store the number of correct predictions we will make
    correct_predictions = 0
    sample_idx = 0  # Track which sample we're processing
    
    for image, target in tqdm(dataloader, desc=label):
        # Map targets in contiguous set starting from zero
        target = torch.Tensor([contig_cat2idx[t.item()] for t in target]).long()

        image = image.to(device)
        target = target.to(device)

        image_features = model.encode_image(image)
        # and normalize
        image_features /= image_features.norm(dim=-1, keepdim=True)
        # For each image in the batch, create text prompts for all categories using that image's color
        batch_text_inputs = []
        for i in range(image.size(0)):
            if sample_idx+i < len(found_colors):
                sample_color = found_colors[sample_idx + i]
            else:
                sample_color = ""
            # Create prompts for all categories using this sample's color
            sample_prompts = [f"a photo of a {CLASS_NAMES[c]}, a {sample_color} colored type of flower." 
                            for c in categories]
            batch_text_inputs.extend(sample_prompts)
        
        # Tokenize all text inputs
        text_inputs = clip.tokenize(batch_text_inputs).to(device)
        # Reshape to (batch_size, num_categories, token_length)
        text_inputs = text_inputs.view(image.size(0), len(categories), -1)


        # Process each image in the batch
        batch_predictions = []
        for i in range(image.size(0)):
            # Get text features for all categories for this specific image
            sample_text_inputs = text_inputs[i]  # (num_categories, token_length)
            text_features = model.encode_text(sample_text_inputs)
            text_features /= text_features.norm(dim=-1, keepdim=True)
            
            # Compute similarity between this image and all category texts
            sample_image_features = image_features[i:i+1]  # Keep batch dimension
            similarities = (sample_image_features @ text_features.T)
            predicted_class = similarities.argmax(dim=-1)
            batch_predictions.append(predicted_class)
        
        predicted_classes = torch.cat(batch_predictions)
        
        # Check which predictions are correct
        correct_predictions += (predicted_classes == target).sum().item()
        
        # Update sample index
        sample_idx += image.size(0)

    # Compute accuracy
    accuracy = correct_predictions / len(dataset)
    return accuracy

In [27]:
# Updated function calls
base_accuracy = eval(model=model, dataset=test_base, categories=base_classes, 
                    batch_size=64, device=device, found_colors=found_colors,
                    label="🧠 Zero-shot evaluation on Base Classes")

print(f"🔍 Base classes accuracy: {base_accuracy*100:.2f}%")

🧠 Zero-shot evaluation on Base Classes: 100%|██████████| 39/39 [08:01<00:00, 12.35s/it]

🔍 Base classes accuracy: 70.40%


In [29]:
novel_accuracy = eval(model=model, dataset=test_novel, categories=novel_classes, 
                     batch_size=32, device=device, found_colors=novel_found_colors,
                     label="🧠 Zero-shot evaluation on Novel Classes")
print(f"🔍 Novel classes accuracy: {novel_accuracy*100:.2f}%")

🧠 Zero-shot evaluation on Novel Classes: 100%|██████████| 115/115 [11:38<00:00,  6.07s/it]

🔍 Novel classes accuracy: 77.23%


In [ ]:
@torch.no_grad() # we don't want gradients
def eval(model, dataset, categories, batch_size, device, label=""):
    # let's set the model in evaluation mode
    model.eval()

    # Remap labels into a contiguous set starting from zero
    contig_cat2idx = {cat: idx for idx, cat in enumerate(categories)}

    # here we apply the standard CLIP template used for oxford flowers to all categories
    # and immediately tokenize each sentence (convert natural language into numbers - feel free to print the text input to inspect them)
    text_inputs = clip.tokenize(
        [f"a photo of a {CLASS_NAMES[c]}, a type of flower." for c in categories]
    ).to(device)

    # we can encode the text features once as they are shared for all images
    # therefore we do it outside the evaluation loop
    text_features = model.encode_text(text_inputs)
    # and here we normalize them (standard pratice with CLIP)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # simple dataloader creation
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # here we store the number of correct predictions we will make
    correct_predictions = 0
    for image, target in tqdm(dataloader, desc=label):
        # base categories range from 0 to 50, whil novel ones from 51 to 101
        # therefore we must map categories to the [0, 50], otherwise we will have wrong predictions
        # Map targets in contiguous set starting from zero
        # Labels needs to be .long() in pytorch
        target = torch.Tensor([contig_cat2idx[t.item()] for t in target]).long()

        image = image.to(device)
        target = target.to(device)

        # forward image through CLIP image encoder
        image_features = model.encode_image(image)
        # and normalize
        image_features /= image_features.norm(dim=-1, keepdim=True)

        # here cosine similarity between image and text features and keep the argmax for every row (every image)
        predicted_class = (image_features @ text_features.T).argmax(dim=-1)
        # now we check which are correct, and sum them (False == 0, True == 1)
        correct_predictions += (predicted_class == target).sum().item()

    # and now we compute the accuracy
    accuracy = correct_predictions / len(dataset)
    return accuracy

base_accuracy = eval(model=model, dataset=test_base, categories=base_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Base Classes")
novel_accuracy = eval(model=model, dataset=test_novel, categories=novel_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Novel Classes")

print()
print(f"🔍 Base classes accuracy: {base_accuracy*100:.2f}%")
print(f"🔍 Novel classes accuracy: {novel_accuracy*100:.2f}%")

🧠 Zero-shot evaluation on Novel Classes: 100%|██████████| 29/29 [01:20<00:00,  2.76s/it]


🔍 Base classes accuracy: 71.29%
🔍 Novel classes accuracy: 78.24%


## Harmonic Mean
Few-Shot Adaptations papers usually report the Harmonic Mean.
The harmonic mean tends to mitigate the impact of large outliers (base accuracy) and aggravate the impact of small ones (novel accuracy).
Thus, achieving very high base accuracies at the expense of the novel accuracy will be penalized by the HM.

In [ ]:
def harmonic_mean(base_accuracy, novel_accuracy):
    numerator = 2
    denominator = 1 / base_accuracy + 1 / novel_accuracy
    hm = numerator / denominator
    return hm

print(f"🔍 Harmonic Mean: {harmonic_mean(base_accuracy, novel_accuracy)*100:.2f}%")

🔍 Harmonic Mean: 74.60%
